# 25. k-Means++
**Use case:** Supply Chain — Choosing distribution center locations

A distributor serving retailers across the south-central US is reviewing its warehouse footprint as leases expire. Too few DCs means long, expensive truck runs and missed next-day deliveries; too many means paying fixed costs for half-empty buildings. The company has every customer's location and annual case volume, plus freight rates and DC operating costs. Success means a recommended number and rough placement of DCs that minimises total annual cost while meeting a next-day service target, ready to be refined against real candidate sites.

> 💡 Imagine placing 3 ice-cream vans on a beach. Instead of dropping them at random (and maybe parking two side by side), you put the first one anywhere, then place each next van where people are farthest from any existing van, with some randomness. Then you let them drift toward their crowds.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## CONCEPT: Seed 3 DC sites the k-means++ way
Step through k-means++ seeding by hand on 12 customer towns and compare it with random seeding over 200 trials.

| 5W + H | |
|---|---|
| **Who** | A network-planning analyst learning why the starting points of k-means matter for warehouse siting. |
| **What** | Picking 3 starting distribution-center locations from 12 customer points, with probability proportional to squared distance (D squared). |
| **When** | Before any k-means iterations: this is only the initialisation step. |
| **Where** | Three customer clusters (west, north-east, south-east) on a simple mile grid, in plain numpy. |
| **Why** | Random starts often put two DCs in the same region and none in another; k-means++ makes such bad starts rare. |
| **How** | Pick the first seed at random, compute each customer's squared distance to its nearest seed, sample the next seed in proportion, and repeat. |

**🧠 ML concepts:** Centroid initialisation, k-means++ D-squared sampling, Inertia (within-cluster sum of squares), Bad local starts, Comparing methods over many random trials  
**🐍 Python concepts:** numpy broadcasting with None axes, Functions with default arguments, while loops, rng.choice with custom probabilities, Lists converted to numpy arrays, Boolean mean as a percentage

In [ ]:
# k-means++ seeding by hand: pick 3 starting warehouse (DC) sites from 12 customer demand points
import numpy as np

# 🐍 PY: each row is a customer town's (x, y) position in miles on a simple map grid
customers = np.array([
    [2, 3], [3, 2], [1, 1], [2, 1],            # cluster near the west
    [20, 21], [22, 19], [21, 22], [19, 20],    # cluster in the north-east
    [30, 4], [31, 2], [29, 3], [32, 5],        # cluster in the south-east
], dtype=float)
K = 3

def dist2_to_nearest(points, centers):
    # 🐍 PY: broadcasting points[:, None, :] - centers[None, :, :] gives all point-center differences at once
    d2 = ((points[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    # 🧠 ML: D(x)^2 = squared distance from each customer to its CLOSEST chosen center so far
    return d2.min(axis=1)

def kmeanspp_seed(points, k, rng, verbose=False):
    # 🧠 ML: step 1 - first center is a uniformly random customer
    centers = [points[rng.integers(len(points))]]
    while len(centers) < k:
        d2 = dist2_to_nearest(points, np.array(centers))
        # 🧠 ML: step 2 - probability of picking a point is proportional to D^2: far points are favoured
        probs = d2 / d2.sum()
        if verbose:
            # 🐍 PY: f"{p:.2f}" formats each probability with 2 decimals
            print("  pick probs:", " ".join(f"{p:.2f}" for p in probs))
        # 🐍 PY: rng.choice with p= draws an index using our custom probabilities
        centers.append(points[rng.choice(len(points), p=probs)])
        if verbose:
            print(f"  chose center #{len(centers)} at {centers[-1]}")
    return np.array(centers)

def inertia(points, centers):
    # 🧠 ML: inertia = total squared distance from customers to their nearest center (lower is better)
    return dist2_to_nearest(points, centers).sum()

rng = np.random.default_rng(42)
print("k-means++ seeding, step by step:")
pp = kmeanspp_seed(customers, K, rng, verbose=True)
print(f"k-means++ seeds:\n{pp}\ninertia = {inertia(customers, pp):.1f}\n")

# 🧠 ML: compare with plain random seeding over many trials - how often do 2 seeds land in one cluster?
rand_scores, pp_scores = [], []
for trial in range(200):
    # 🐍 PY: rng.choice(..., replace=False) picks K different row indexes
    rand_scores.append(inertia(customers, customers[rng.choice(len(customers), K, replace=False)]))
    pp_scores.append(inertia(customers, kmeanspp_seed(customers, K, rng)))
# 🐍 PY: converting a list to np.array lets us do vectorised comparisons like (arr > 100).mean()
rand_scores, pp_scores = np.array(rand_scores), np.array(pp_scores)
print("Seeding only (before any k-means iterations), 200 trials:")
print(f"  random    : mean inertia {rand_scores.mean():7.1f}, bad starts (>100) {(rand_scores > 100).mean():.0%}")
print(f"  k-means++ : mean inertia {pp_scores.mean():7.1f}, bad starts (>100) {(pp_scores > 100).mean():.0%}")

**Try this next:**
- Set K = 4 and see where the extra seed tends to land.
- Use probabilities proportional to D (not D squared) and compare the bad-start rate.
- Move the south-east cluster closer to the west one and rerun.

## BUILD: Random vs k-means++ across 10 seeds
Cluster 3,000 customer locations into 8 DC sites and measure how initialisation affects inertia, iterations and bad solutions.

| 5W + H | |
|---|---|
| **Who** | A supply chain analyst proposing 8 regional DC sites across the south-central US. |
| **What** | KMeans with init='random' vs init='k-means++' (one start each) over 10 seeds, then a final 10-restart k-means++ solution. |
| **When** | During a network-design study, before detailed site and cost analysis. |
| **Where** | 3,000 synthetic customers around 8 metros, converted from lat/lon to approximate miles. |
| **Why** | A bad initialisation can leave a whole metro without a nearby DC; the analyst needs to know how reliable the method is. |
| **How** | Project to miles, hold out 20% of customers, run both inits across seeds, summarise with pandas, then report the recommended sites. |

**🧠 ML concepts:** Effect of initialisation on k-means, Local optima, Inertia and n_iter_, Holdout check for clustering, Multiple restarts (n_init)  
**🐍 Python concepts:** pandas DataFrame creation and describe(), np.column_stack, Nested loops building a list of dicts, DataFrame.pivot, groupby with named aggregations, assign and boolean columns

In [ ]:
# k-means vs k-means++ initialisation for choosing distribution-center (DC) sites from customer locations
import warnings
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)

# ---- Synthetic customers: 3,000 store/delivery points around 8 metro areas (lat/lon in degrees) ----
metros = pd.DataFrame({
    "metro": ["Dallas", "Houston", "Oklahoma City", "Tulsa", "Kansas City", "Denver", "Memphis", "Little Rock"],
    "lat": [32.78, 29.76, 35.47, 36.15, 39.10, 39.74, 35.15, 34.75],
    "lon": [-96.80, -95.37, -97.52, -95.99, -94.58, -104.99, -90.05, -92.29],
    "share": [0.22, 0.20, 0.10, 0.08, 0.12, 0.12, 0.09, 0.07],
})
N = 3000
# 🐍 PY: rng.choice with p= picks a metro index for each customer, weighted by market share
idx = rng.choice(len(metros), N, p=metros.share)
customers = pd.DataFrame({
    "customer_id": np.arange(N),
    "lat": metros.lat.to_numpy()[idx] + rng.normal(0, 0.35, N),
    "lon": metros.lon.to_numpy()[idx] + rng.normal(0, 0.45, N),
    "weekly_cases": rng.lognormal(3.0, 0.6, N).round(),
})
print(customers.describe().loc[["mean", "min", "max"], ["lat", "lon", "weekly_cases"]].round(2))

# 🧠 ML: convert degrees to approximate miles so 1 unit east = 1 unit north (k-means uses straight distance)
# 🐍 PY: np.cos needs radians, so np.radians converts the average latitude first
MI_PER_DEG_LAT = 69.0
mi_per_deg_lon = 69.0 * np.cos(np.radians(customers.lat.mean()))
X = np.column_stack([customers.lat * MI_PER_DEG_LAT, customers.lon * mi_per_deg_lon])
# 🧠 ML: hold out 20% of customers to check that DC sites also serve customers we did not fit on
X_fit, X_new = train_test_split(X, test_size=0.2, random_state=42)

K = 8
rows = []
for seed in range(10):
    for init in ("random", "k-means++"):
        # 🧠 ML: n_init=1 shows the effect of ONE initialisation; production uses several and keeps the best
        km = KMeans(n_clusters=K, init=init, n_init=1, random_state=seed, max_iter=300).fit(X_fit)
        # 🧠 ML: inertia_ = sum of squared miles from each customer to its DC; score() on new data = -inertia
        rows.append({"seed": seed, "init": init, "inertia_M": km.inertia_ / 1e6,
                     "iterations": km.n_iter_, "holdout_inertia_M": -km.score(X_new) / 1e6})
res = pd.DataFrame(rows)

# 🐍 PY: pivot turns long rows into a seed x init table for side-by-side comparison
print("\nTraining inertia (millions of sq-miles) per seed:")
print(res.pivot(index="seed", columns="init", values="inertia_M").round(2).to_string())
# 🐍 PY: groupby + agg computes several summaries per init method in one go
summary = res.groupby("init").agg(mean_inertia=("inertia_M", "mean"), worst=("inertia_M", "max"),
                                  sd=("inertia_M", "std"), avg_iters=("iterations", "mean"),
                                  holdout=("holdout_inertia_M", "mean"))
print("\nSummary over 10 seeds:")
print(summary.round(2).to_string())

best_inertia = res.inertia_M.min()
# 🧠 ML: a "bad" run converged to a local optimum >5% worse than the best found - e.g. 2 DCs in one metro
bad = res.assign(bad=res.inertia_M > best_inertia * 1.05).groupby("init").bad.mean()
print(f"\nShare of runs >5% worse than best: random {bad['random']:.0%}, k-means++ {bad['k-means++']:.0%}")

# Final DC sites from the recommended setting (k-means++, 10 restarts)
final = KMeans(n_clusters=K, init="k-means++", n_init=10, random_state=42).fit(X)
sites = pd.DataFrame(final.cluster_centers_, columns=["lat", "lon"])
sites["lat"] /= MI_PER_DEG_LAT
sites["lon"] /= mi_per_deg_lon
sites["customers"] = np.bincount(final.labels_)
# 🧠 ML: average straight-line distance from each customer to its assigned DC (not squared)
d = np.linalg.norm(X - final.cluster_centers_[final.labels_], axis=1)
sites["avg_miles"] = pd.Series(d).groupby(final.labels_).mean().to_numpy()
print("\nRecommended DC sites (k-means++, n_init=10):")
print(sites.round(2).sort_values("customers", ascending=False).to_string(index=False))
print(f"\nBusiness read-out: average customer is {d.mean():.0f} miles from its DC. Random init often"
      f"\nputs two DCs in one metro and leaves another metro unserved - k-means++ makes that rare.")

**Try this next:**
- Set n_init=5 in the loop for both inits: does random catch up?
- Change K to 6 and see which metros share a DC.
- Remove the miles conversion (use raw degrees) and compare the site table.

## PRODUCTION: Demand-weighted DC network sized by total cost
Use demand-weighted k-means++ to place DCs, choose k by fixed plus transport cost in haversine road miles, check stability and report the network.

| 5W + H | |
|---|---|
| **Who** | A network strategy team preparing a DC footprint recommendation for the CFO. |
| **What** | Weighted k-means++ for k = 2 to 12, each network costed with real great-circle miles, a road circuity factor, fixed DC cost and a 150-mile service check. |
| **When** | During annual network-design reviews or before a major lease decision. |
| **Where** | 4,000 customers around 12 metros with annual case volumes; runs in seconds with scikit-learn. |
| **Why** | More DCs cut transport cost but add fixed cost; the right number of DCs is a cost and service trade-off, not a statistics question. |
| **How** | sample_weight = annual cases, assign customers by haversine distance, compute total cost per k, add a service-level constraint, check seed stability, and save. |

**🧠 ML concepts:** Weighted clustering with sample_weight, Choosing k by a business objective, Haversine distance, Service-level constraint, Stability via adjusted Rand index, Limits of squared-distance objectives  
**🐍 Python concepts:** Classes with helper methods, map() over a tuple, Broadcasting [:, None] for an all-pairs matrix, List comprehension filters, groupby.agg with lambda, joblib.dump

In [ ]:
# Production-style DC network design: demand-weighted k-means++ (sample_weight), choose k by total
# annual cost (fixed DC cost + transport cost on haversine miles), stability check, site report, save.
import os
import tempfile
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

warnings.filterwarnings("ignore")
EARTH_RADIUS_MI = 3958.8
FIXED_COST_PER_DC = 1_400_000      # $/year: lease, staff, WMS licence for one DC
COST_PER_CASE_MILE = 0.012         # $ per case per mile (truckload + last mile blended)
CIRCUITY = 1.20                    # road miles are ~20% longer than straight-line miles
SERVICE_RADIUS_MI = 150            # next-day delivery promise


def make_customers(n=4000, seed=42):
    rng = np.random.default_rng(seed)
    metros = pd.DataFrame({
        "lat": [32.78, 29.76, 35.47, 36.15, 39.10, 39.74, 35.15, 34.75, 30.27, 38.63, 41.26, 37.69],
        "lon": [-96.80, -95.37, -97.52, -95.99, -94.58, -104.99, -90.05, -92.29, -97.74, -90.20,
                -95.93, -97.34],
        "pop_share": [.17, .16, .07, .06, .09, .10, .06, .04, .09, .08, .04, .04],
    })
    idx = rng.choice(len(metros), n, p=metros.pop_share)
    # 🧠 ML: big metros also order more per customer - demand is NOT spread evenly across points
    size_boost = 1 + 3 * metros.pop_share.to_numpy()[idx]
    return pd.DataFrame({
        "lat": metros.lat.to_numpy()[idx] + rng.normal(0, 0.4, n),
        "lon": metros.lon.to_numpy()[idx] + rng.normal(0, 0.5, n),
        "annual_cases": (rng.lognormal(6.5, 0.8, n) * size_boost).round(),
    })


def haversine_mi(lat1, lon1, lat2, lon2):
    # 🧠 ML: great-circle distance on a sphere - the honest "as the crow flies" miles between two lat/lons
    # 🐍 PY: np.radians works element-wise on whole arrays, so this computes thousands of distances at once
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * EARTH_RADIUS_MI * np.arcsin(np.sqrt(a))


class DCPlanner:
    """Clusters customers in an equal-area projection, then costs the network with haversine miles."""

    def __init__(self, customers):
        self.c = customers
        self.lat0 = customers.lat.mean()
        # 🧠 ML: k-means needs Euclidean space; this local projection makes 1 unit ~= 1 mile both ways
        self.X = np.column_stack([customers.lat * 69.0,
                                  customers.lon * 69.0 * np.cos(np.radians(self.lat0))])

    def to_latlon(self, centers):
        return centers[:, 0] / 69.0, centers[:, 1] / (69.0 * np.cos(np.radians(self.lat0)))

    # 🧠 ML: caveat - k-means minimises SQUARED distance; transport cost is linear in miles, so the
    # 🧠 ML: centroids are near-optimal, not optimal (a Weiszfeld / MIP step can refine them)
    def fit(self, k, weighted=True, seed=42):
        # 🧠 ML: sample_weight = annual cases -> centroids are pulled toward heavy-demand customers
        w = self.c.annual_cases.to_numpy() if weighted else None
        km = KMeans(n_clusters=k, init="k-means++", n_init=5, random_state=seed)
        return km.fit(self.X, sample_weight=w)

    def cost(self, km):
        dc_lat, dc_lon = self.to_latlon(km.cluster_centers_)
        # 🧠 ML: re-assign each customer to the nearest DC by REAL miles (haversine), not projected units
        d = haversine_mi(self.c.lat.to_numpy()[:, None], self.c.lon.to_numpy()[:, None],
                         dc_lat[None, :], dc_lon[None, :])
        assign = d.argmin(axis=1)
        miles = d[np.arange(len(d)), assign] * CIRCUITY
        transport = float((miles * self.c.annual_cases * COST_PER_CASE_MILE).sum())
        fixed = FIXED_COST_PER_DC * km.n_clusters
        service = float(np.average(miles <= SERVICE_RADIUS_MI, weights=self.c.annual_cases))
        return {"k": km.n_clusters, "fixed_M": fixed / 1e6, "transport_M": transport / 1e6,
                "total_M": (fixed + transport) / 1e6, "service_pct": 100 * service,
                "assign": assign, "miles": miles, "dc_lat": dc_lat, "dc_lon": dc_lon}


if __name__ == "__main__":
    cust = make_customers()
    planner = DCPlanner(cust)
    print(f"{len(cust)} customers, {cust.annual_cases.sum() / 1e6:.2f}M cases/year")

    print("\n== Choosing k by total annual cost (demand-weighted k-means++) ==")
    print("  k  fixed$M  transport$M  total$M  served<150mi")
    results = []
    for k in range(2, 13):
        r = planner.cost(planner.fit(k))
        results.append(r)
        bar = "#" * int((r["total_M"] - 10) * 2) if r["total_M"] > 10 else ""
        print(f" {k:2d}  {r['fixed_M']:7.1f}  {r['transport_M']:11.2f}  {r['total_M']:7.2f}   "
              f"{r['service_pct']:5.1f}%  {bar}")
    # 🧠 ML: the elbow is a business trade-off - each extra DC adds fixed cost but cuts transport cost
    best = min(results, key=lambda r: r["total_M"])
    k_best = best["k"]
    print(f"  -> cheapest network: k={k_best} at ${best['total_M']:.2f}M/year")

    # 🧠 ML: if business requires a service level, pick the cheapest k that meets it (a constraint)
    meets = [r for r in results if r["service_pct"] >= 95]
    if meets:
        k_sla = min(meets, key=lambda r: r["total_M"])
        print(f"  -> cheapest k meeting 95% within {SERVICE_RADIUS_MI} mi: k={k_sla['k']}"
              f" at ${k_sla['total_M']:.2f}M/year")

    print(f"\n== Weighted vs unweighted clustering at k={k_best} ==")
    unw = planner.cost(planner.fit(k_best, weighted=False))
    print(f"  unweighted transport ${unw['transport_M']:.2f}M  vs  weighted ${best['transport_M']:.2f}M"
          f"  (saving ${1e3 * (unw['transport_M'] - best['transport_M']):.0f}K/year)")

    # 🧠 ML: stability - do different random seeds give the same customer-to-DC assignments?
    labels = [planner.fit(k_best, seed=s).labels_ for s in range(4)]
    ari = [adjusted_rand_score(labels[0], l) for l in labels[1:]]
    print(f"\nStability across seeds (adjusted Rand index, 1.0 = identical): {np.round(ari, 3).tolist()}")

    print(f"\n== Recommended DC sites (k={k_best}) ==")
    # 🐍 PY: building a DataFrame from a dict of arrays, then groupby on the assignment column
    rep = pd.DataFrame({"dc": best["assign"], "cases": cust.annual_cases, "miles": best["miles"]})
    site = rep.groupby("dc").agg(customers=("cases", "size"), cases_K=("cases", lambda s: s.sum() / 1e3),
                                 avg_road_mi=("miles", "mean"), max_road_mi=("miles", "max"))
    site["lat"], site["lon"] = best["dc_lat"].round(2), best["dc_lon"].round(2)
    site["demand_share"] = (100 * site.cases_K / site.cases_K.sum()).round(1)
    print(site.sort_values("cases_K", ascending=False).round(1).to_string())

    print("\nModel-risk note: centroids are a STARTING point - snap each to real candidate sites (labour,"
          "\nzoning, interstate access, incentives) and re-run cost with a road-network router before"
          "\ncommitting capital. Re-validate when demand shifts >10% in any region.")

    path = os.path.join(tempfile.mkdtemp(), "dc_plan.joblib")
    # 🐍 PY: joblib.dump stores the fitted KMeans plus the chosen sites as one bundle
    joblib.dump({"kmeans": planner.fit(k_best), "sites": site.reset_index().to_dict("records")}, path)
    print(f"Saved model + site table -> {os.path.basename(path)} in a temp dir")

**Try this next:**
- Halve FIXED_COST_PER_DC and see how many DCs become optimal.
- Set SERVICE_RADIUS_MI to 100 and find the new cheapest compliant k.
- Give one metro 5x demand and see whether a centroid moves toward it.